# LangChain Practice: Beginner to Full Pipeline

This notebook teaches LangChain step-by-step and then maps directly to `langchainfull.py`.

## What you will learn
1. Models (LLM / Chat Model)
2. Prompts
3. Output Parsers
4. LCEL Chains
5. Tools
6. Memory
7. Documents, Embeddings, Vector Store, Retriever
8. RAG (Retrieval-Augmented Generation)
9. Full conversational pipeline

In [1]:
# Step 0: Install dependencies (run once)
# !pip install -U langchain langchain-core langchain-community langchain-text-splitters
# !pip install -U langchain-google-genai python-dotenv faiss-cpu

In [7]:
# Step 1: Imports
import os
import re
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.tools import tool
from langchain_core.documents import Document

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [18]:
# Step 2: Environment + model setup
load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    raise ValueError("GEMINI_API_KEY not found. Put it in .env file.")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0.3,
    max_tokens=256,
)

print("Model ready: gemini-2.5-flash-lite")

Model ready: gemini-2.5-flash-lite


## Concept 1: PromptTemplate
Prompt templates help you reuse prompt structures with variables.

In [12]:
prompt = PromptTemplate.from_template("Explain {topic} for a beginner. Key Point : {key_point}")
formatted = prompt.format(topic="Python programming", key_point="Tkinter library for GUI")
print(formatted)

Explain Python programming for a beginner. Key Point : Tkinter library for GUI


## Concept 2: LLM / Chat Model
A chat model takes text input and returns generated text.

In [13]:
response = llm.invoke("What is LangChain in one sentence?")
print(response.content)

LangChain is a framework that helps developers build applications powered by large language models by providing tools for chaining together different components.


## Concept 3: Output Parser
Parsers normalize model output into a useful format (here: plain string).

In [19]:
parser = StrOutputParser()
simple_chain = prompt | llm | parser
print(simple_chain.invoke({"topic": "Python", "key_point": "Machine Learning"}))

Let's dive into Python, especially with a focus on how it's a fantastic tool for **Machine Learning**!

Imagine you want to teach a computer to do something smart, like recognize a cat in a picture, predict house prices, or recommend movies you might like. That's where Machine Learning (ML) comes in, and Python is your go-to language for it.

## What is Python?

Think of Python as a **super-friendly and readable language** that humans can easily understand, and that computers can also understand. It's like a set of instructions you give to your computer.

Here's why it's great for beginners:

*   **Easy to Read:** Python's syntax (the way you write code) is very similar to English. This makes it much less intimidating than some other programming languages.
*   **Versatile:** You can use Python for almost anything: building websites, automating tasks, analyzing data, and, of course, Machine Learning.
*   **Huge Community:** Lots of people use Python, so if you get stuck, there are tons 

## Concept 4: Tools
Tools are functions your app/agent can call for specific tasks.

In [ ]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b

print(multiply.invoke({"a": 7, "b": 8}))

## Concept 5: Documents -> Embeddings -> Vector Store -> Retriever
This is the core retrieval pipeline used in RAG systems.

In [ ]:
docs = [
    Document(page_content="LangChain is a framework for building LLM-powered apps."),
    Document(page_content="RAG combines retrieval with generation for better factual answers."),
    Document(page_content="Embeddings convert text into vectors for semantic search."),
]

splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
splits = splitter.split_documents(docs)

embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vectorstore = FAISS.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

print("Chunks:", len(splits))
print("Retriever ready")

## Concept 6: RAG Chain
RAG retrieves relevant context and then asks the model to answer with that context.

In [ ]:
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "Given chat history and latest user question, rewrite it as a standalone question."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

history_aware_retriever = create_history_aware_retriever(llm, retriever, contextualize_q_prompt)

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the context to answer. If not found, say you do not know.\n\nContext:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

qa_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_chain)

print("RAG chain ready")

## Concept 7: Memory (Multi-turn conversation)
RunnableWithMessageHistory helps maintain memory across turns with session IDs.

In [ ]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

conversational_rag = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)

print("Conversation memory ready")

## Concept 8: Full Pipeline Function
This function combines tools + conversational RAG in one place.

In [ ]:
def try_math_tool(user_input: str):
    # Very simple parser for patterns like '12 * 9'
    match = re.search(r"(\d+)\s*\*\s*(\d+)", user_input)
    if not match:
        return None
    a, b = int(match.group(1)), int(match.group(2))
    return multiply.invoke({"a": a, "b": b})

def run_pipeline(user_input: str, session_id: str = "demo"):
    tool_result = try_math_tool(user_input)
    if tool_result is not None:
        return f"[Tool:multiply] {tool_result}"

    result = conversational_rag.invoke(
        {"input": user_input},
        config={"configurable": {"session_id": session_id}}
    )
    return result["answer"]

print(run_pipeline("What is LangChain?", session_id="s1"))
print(run_pipeline("What does RAG mean?", session_id="s1"))
print(run_pipeline("15 * 6", session_id="s1"))

## Concept Map (Quick Summary)
- Model: Generates text
- Prompt: Structures instructions
- Parser: Formats output
- Chain (LCEL): Connects components with `|`
- Tool: External function/action
- Embeddings + Vector Store + Retriever: Search relevant knowledge
- RAG: Retriever + Model answer
- Memory: Keeps conversation context by session
- Pipeline: Orchestrates all pieces in one app